# Induction Evals
We evaluate that the induction case is strong enough to drive a wedge between LLM performance in this notebook.

In [9]:
"""The following generates the Quiz all our models will be evaluated on."""

import itertools
import string
import numpy as np
import yaml

import logging

from ordered_set import OrderedSet
from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO)
load_dotenv("./keys.env", verbose=True)

from smolbench.induction.chromatic import (
    ChromaticIntervalsConfig,
    Prompter,
    get_random_exclusive_quiz,
    anneal_intervals,
)
from smolbench.evals import Marks
from smolbench.evals.openrouter import evaluate

template = string.Template(
    "You are a Boolean classifier.\n"
    "\n"
    "Task: determine whether the statement in the Question is logically "
    "possible given the Context.\n"
    "\n"
    "Output format:\n"
    "Return exactly one of these two strings and nothing else:\n"
    "True\n"
    "False\n"
    "\n"
    "Do not output any explanation, punctuation, quotes, labels, code fences, "
    "or extra whitespace."
    "Stop immediately after writing True or False."
    "\n"
    "Context:\n"
    "There is a ceremonial role called the $role, whose job it is to"
    " head the $parade parade. No one else besides the $role is able to head"
    " the $parade parade. At the end of one's term as $role, they have a ceremony where they hand off the"
    " $role ceremonial sceptre to their successor. The following lists the people who were $role and"
    " the years they were $role:\n"
    "$positive_info\n"
    "\n"
    "Question:\n"
    "Has $color1 handed the sceptre to $color2?"
)

extens_template = string.Template(
    "You are a Boolean classifier.\n"
    "\n"
    "Task: determine whether the statement in the Question is logically "
    "possible given the Context.\n"
    "\n"
    "Output format:\n"
    "Return exactly one of these two strings and nothing else:\n"
    "True\n"
    "False\n"
    "\n"
    "Do not output any explanation, punctuation, quotes, labels, code fences, "
    "or extra whitespace."
    "Stop immediately after writing True or False."
    "\n"
    "Context:\n"
    "There is a ceremonial role called the $role, whose job it is to"
    " head the $parade parade. No one else besides the $role is able to head"
    " the $parade parade. At the end of one's term as $role, they have a ceremony where they hand off the"
    " $role ceremonial sceptre to their successor. The following lists each year and who was $role"
    " that year:\n"
    "$positive_info\n"
    "\n"
    "Question:\n"
    "Has $color1 handed the sceptre to $color2?"
)


def query_gen(
    labels_to_intervals: Dict[Color, Intervals],
    interval_to_label: Dict[Interval, Color],
    seed: int,
) -> Iterable[Tuple[Dict[str, str], bool]]:
    """Generates direct-succession queries.

    Yields True for each unique (predecessor, successor) pair and an equal
    number of False for randomly-sampled non-successor pairs.
    """
    rng: np.random.Generator = np.random.default_rng(seed)
    sorted_intervals = sorted(interval_to_label.items(), key=lambda item: item[0][0])
    # True cases: pairs where color2 immediately followed color1.
    true_pairs: OrderedSet = OrderedSet(
        (c1, c2)
        for ((_s1, e1), c1), ((s2, _e2), c2) in zip(sorted_intervals, sorted_intervals[1:])
    )
    for color1, color2 in true_pairs:
        yield {"color1": color1, "color2": color2}, True
    # False cases: same count, randomly sampled non-successor pairs.
    all_colors: list = list(labels_to_intervals.keys())
    false_pairs: OrderedSet = OrderedSet()
    while len(false_pairs) < len(true_pairs):
        c1, c2 = (str(c) for c in rng.choice(all_colors, size=2, replace=False))
        if (c1, c2) not in true_pairs:
            false_pairs.add((c1, c2))
    for color1, color2 in false_pairs:
        yield {"color1": color1, "color2": color2}, False


SEED: int = 1776
intens_quiz, extens_quiz = get_random_exclusive_quiz(
    ChromaticIntervalsConfig(
        n=int(36 * 250),
        intervals= 250 // 4,
        colors=45,
        seed=SEED,
    ),
    Prompter(
        template,
        {
            "role": "Twislax",
            "parade": "Gildane",
        },
        query_gen,
        extens_template,
    ),
)

## Prompt Validation

In [10]:
print(intens_quiz[0].prompt)

You are a Boolean classifier.

Task: determine whether the statement in the Question is logically possible given the Context.

Output format:
Return exactly one of these two strings and nothing else:
True
False

Do not output any explanation, punctuation, quotes, labels, code fences, or extra whitespace.Stop immediately after writing True or False.
Context:
There is a ceremonial role called the Twislax, whose job it is to head the Gildane parade. No one else besides the Twislax is able to head the Gildane parade. At the end of one's term as Twislax, they have a ceremony where they hand off the Twislax ceremonial sceptre to their successor. The following lists the people who were Twislax and the years they were Twislax:
qo was Twislax on 1139 to 1369.
Lw was Twislax on 3995 to 4228 and 5238 to 5371.
Gv was Twislax on 1369 to 1616 and 4872 to 5017.
Wd was Twislax on 2528 to 2835.
UE was Twislax on 6721 to 7122.
ld was Twislax on 8096 to 8254.
Gt was Twislax on 3444 to 3785.
nR was Twisla

In [11]:
print(extens_quiz[0].prompt)

You are a Boolean classifier.

Task: determine whether the statement in the Question is logically possible given the Context.

Output format:
Return exactly one of these two strings and nothing else:
True
False

Do not output any explanation, punctuation, quotes, labels, code fences, or extra whitespace.Stop immediately after writing True or False.
Context:
There is a ceremonial role called the Twislax, whose job it is to head the Gildane parade. No one else besides the Twislax is able to head the Gildane parade. At the end of one's term as Twislax, they have a ceremony where they hand off the Twislax ceremonial sceptre to their successor. The following lists each year and who was Twislax that year:
Year 0: Ef.
Year 1: Ef.
Year 2: Ef.
Year 3: Ef.
Year 4: Ef.
Year 5: Ef.
Year 6: Ef.
Year 7: Ef.
Year 8: Ef.
Year 9: Ef.
Year 10: Ef.
Year 11: Ef.
Year 12: Ef.
Year 13: Ef.
Year 14: Ef.
Year 15: Ef.
Year 16: Ef.
Year 17: Ef.
Year 18: Ef.
Year 19: Ef.
Year 20: Ef.
Year 21: Ef.
Year 22: Ef.
Ye

# Decoder-Only Model
This section tests classical decoder-only models.

In [12]:
decode_intens_eval: Marks = evaluate(
    intens_quiz, "mistralai/devstral-small", SEED
)

In [13]:
decode_extens_eval: Marks = evaluate(
    extens_quiz, "mistralai/devstral-small", SEED
)

In [14]:
# Prints out results.
print(
    decode_intens_eval.correct, decode_intens_eval.incorrect, decode_intens_eval.invalid
)
print(
    decode_extens_eval.correct, decode_extens_eval.incorrect, decode_extens_eval.invalid
)
# Serializes results
with open('decode_intens.yaml', 'w') as file:
    yaml.dump(
        decode_intens_eval, file, default_flow_style=False, indent=4
    )
with open('decode_extens.yaml', 'w') as file:
    yaml.dump(
        decode_extens_eval, file, default_flow_style=False, indent=4
    )

86 34 0
63 57 0


## CoT Model
This section tests a CoT model.

In [15]:
cot_intens_eval: Marks = evaluate(
    intens_quiz, "google/gemma-3-27b-it", SEED, extra_args={"reasoning": {"enabled": True}}
)

In [ ]:
cot_extens_eval: Marks = evaluate(
    extens_quiz, "google/gemma-3-27b-it", SEED, extra_args={"reasoning": {"enabled": True}}
)

In [ ]:
print(cot_intens_eval.correct, cot_intens_eval.incorrect, cot_intens_eval.invalid)
print(cot_extens_eval.correct, cot_extens_eval.incorrect, cot_extens_eval.invalid)
# Serializes results
with open('cot_intens.yaml', 'w') as file:
    yaml.dump(
        cot_intens_eval, file, default_flow_style=False, indent=4
    )
with open('cot_extens.yaml', 'w') as file:
    yaml.dump(
        cot_extens_eval, file, default_flow_style=False, indent=4
    )

86 34 0
78 42 0


## MoE Model
This section tests an MoE model.

In [ ]:
moe_intens_eval: Marks = evaluate(intens_quiz, "qwen/qwen3-30b-a3b-instruct-2507", SEED)

In [ ]:
moe_extens_eval: Marks = evaluate(extens_quiz, "qwen/qwen3-30b-a3b-instruct-2507", SEED)

In [ ]:
print(moe_intens_eval.correct, moe_intens_eval.incorrect, moe_intens_eval.invalid)
print(moe_extens_eval.correct, moe_extens_eval.incorrect, moe_extens_eval.invalid)
# Serializes results
with open('moe_intens.yaml', 'w') as file:
    yaml.dump(
        moe_intens_eval, file, default_flow_style=False, indent=4
    )
with open('moe_extens.yaml', 'w') as file:
    yaml.dump(
        moe_extens_eval, file, default_flow_style=False, indent=4
    )

77 43 0
77 43 0


# HRM Model
This section tests an HRM model.

In [ ]:
# hrm_intens_eval: Marks = evaluate(intens_quiz, "perplexity/pplx-embed-v1-4b", SEED)

In [ ]:
# hrm_extens_eval: Marks = evaluate(extens_quiz, "perplexity/pplx-embed-v1-4b", SEED)

In [ ]:
# print(hrm_intens_eval.correct, hrm_intens_eval.incorrect, hrm_intens_eval.invalid)
# print(hrm_extens_eval.correct, hrm_extens_eval.incorrect, hrm_extens_eval.invalid)
# # Serializes results
# with open('hrm_intens.yaml', 'w') as file:
#     yaml.dump(
#         hrm_intens_eval, file, default_flow_style=False, indent=4
#     )
# with open('cot_extens.yaml', 'w') as file:
#     yaml.dump(
#         hrm_extens_eval, file, default_flow_style=False, indent=4
#     )